# Differentiable loops

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/tutorial/advanced/05_differentiable.ipynb)

Gradients through whole closed loops: a rollout's cost differentiated with JAX, plant and controller co-designed, plant families in one call, and a neural loop inside an MPC.

In [ ]:
# Local conda: minilink already installed. Colab: clone + path + meshcat.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")
    get_ipython().system("pip install -q meshcat")

## 1. A gradient through a closed loop

Every system compiles to JAX, and a JAX rollout is a pure function of the parameters. So the
cost of a whole simulated run can be differentiated with respect to any number in the loop,
a controller gain or a plant's mass, in one call.

The loop: a PD law holding an inverted pendulum, started at 0.4 rad. `loop.compile(
backend="jax")` returns an evaluator whose `rollout(x0, u_seq, t0, dt, params,
n_steps=)` integrates the loop with RK4; its `params` is the loop's tree, one entry per
block. The cost of the run is

$$J(p) = \sum_k \theta_k^2\, \Delta t.$$

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np

from minilink import PD, InvertedPendulum

plant = InvertedPendulum()
loop = PD(Kp=12.0, Kd=2.0, tau=0.05) @ plant
evaluator = loop.compile(backend="jax")

x0 = jnp.array([0.0, 0.4, 0.0])  # the PD filter state, theta, dtheta
dt = 0.01
n_steps = 400


def rollout_cost(params):
    xs = evaluator.rollout(x0, None, 0.0, dt, params, n_steps=n_steps)
    theta = xs[:, 1]

    J = dt * jnp.sum(theta**2)

    return J


params = jax.tree_util.tree_map(jnp.asarray, loop.params)
J, grads = jax.value_and_grad(rollout_cost)(params)
print("J =", float(J))
print("dJ/dKp =", float(grads["controller"]["Kp"][0]))
print("dJ/dm  =", float(grads["plant"]["m"]))

The gradient has the shape of the tree: one entry per leaf, through the controller, the
plant and every integration step. It is exact; a finite difference on $K_p$ agrees.

In [ ]:
eps = 1e-4
shifted = jax.tree_util.tree_map(lambda leaf: leaf, params)
shifted["controller"]["Kp"] = params["controller"]["Kp"] + eps
float((rollout_cost(shifted) - J) / eps)

## 2. Co-design: plant and controller numbers together

The same gradient drives an optimizer. A problem compiled on JAX,
`problem.compile(law % dt, backend="jax")`, scores the loop; `program(wrt)` turns its cost
into a `MathematicalProgram` over the named leaves, with its exact gradient; `bounds` keeps
a design number in range. Here the pendulum's length $l \in [0.5, 1.5]$ m, a design
choice, is optimized with the PD gains.

In [ ]:
from minilink import Optimizer, PlanningProblem, QuadraticCost

cost = QuadraticCost.from_system(plant, Q=np.diag([10.0, 1.0]), R=0.1 * np.eye(1))
problem = PlanningProblem(plant, cost=cost, x0=np.array([0.4, 0.0]), horizon=4.0)
problem_evaluator = problem.compile(PD(Kp=20.0, Kd=4.0) % 0.01, backend="jax")
print(problem_evaluator.evaluate())

wrt = ["plant.l", "controller.Kp", "controller.Kd"]
program = problem_evaluator.program(wrt, bounds={"plant.l": (0.5, 1.5)})
codesign = Optimizer(program, method="auto").solve()
print(codesign)

The optimizer shortens the pendulum to its bound: less gravity torque to fight. Against the
best gains for the original pendulum, the design and the controller found together cost
about half as much.

In [ ]:
gains_only = Optimizer(
    problem_evaluator.program(["controller.Kp", "controller.Kd"]), method="auto"
).solve()
gains_only.cost, codesign.cost

## 3. A family of plants in one call

`rollout_batch` runs one rollout per row of its starts, and a leaf of `params` given as an
array of values runs one plant per value: one vectorized call (`jax.vmap`) for a whole
family. Here the same PD loop on seven pendulum lengths: the gains that hold the short
pendulums lose the long ones, whose gravity torque $mgl\sin\theta$ outgrows $K_p$.

In [ ]:
import matplotlib.pyplot as plt

lengths = jnp.linspace(0.5, 2.0, 7)
family = dict(params, plant=dict(params["plant"], l=lengths))
starts = jnp.tile(x0, (len(lengths), 1))

xs = evaluator.rollout_batch(starts, n_steps=n_steps, dt=dt, params=family)
t = dt * np.arange(n_steps + 1)

for x, length in zip(xs, lengths):
    plt.plot(t, x[:, 1], label=f"l = {length:.2f} m")
plt.xlabel("t [s]")
plt.ylabel("theta [rad]")
plt.legend(fontsize=7)
plt.show()

## 4. The differentiable double loop

Nothing above is special to a PD law. Here an inner loop closes a small neural network
$u = \mathrm{MLP}(x)$ on a hanging pendulum, released at 1 rad, with a free input $d$ added to its command; an outer
MPC plans $d$ on that inner loop with 5 Adam steps per tick (a fixed count, so the whole
loop stays one pure function), its setpoint bound to a port `r`.

In [ ]:
from minilink import Pendulum, TrajectoryOptimizationPlanner, closed_loop
from minilink.blocks import MLP
from minilink.control import ModelPredictiveController

pendulum = Pendulum()
pendulum.x0 = np.array([1.0, 0.0])
net = MLP(input_dim=2, output_dim=1, hidden=(4,), output_gain=0.1).rename_ports(u="x")
inner = closed_loop(net, pendulum, reference=False, disturbance=True)
inner.inputs["d"].lower = np.array([-2.0])
inner.inputs["d"].upper = np.array([2.0])

inner_cost = QuadraticCost.from_system(inner, Q=np.diag([1.0, 0.1]), R=0.01 * np.eye(1))
outer_planner = TrajectoryOptimizationPlanner(
    PlanningProblem(
        inner, cost=inner_cost, x0=pendulum.x0, horizon=0.5, U=inner.input_box
    ),
    n_steps=5,
    transcription="single_shooting",
    backend="jax",
    method="adam",
    n_iterations=5,
)
mpc = ModelPredictiveController(
    outer_planner, inputs={"x0": "y", "cost.x_bar": "r"}
)

The double loop compiles once on JAX. One program reaches a plant parameter, a network
bias and an MPC weight, three levels deep, and Adam optimizes them together. The MPC's
input weight $R$ is bounded below: a weight stays positive (unbounded, the tuning drives it
negative), and Adam projects each step onto the bounds.

In [ ]:
double_problem = PlanningProblem(inner, cost=inner_cost, x0=pendulum.x0, horizon=2.0)
double_evaluator = double_problem.compile(mpc % 0.1, backend="jax")

wrt = ["plant.plant.m", "plant.controller.b1", "controller.solve.cost.R"]
start = double_evaluator.params
bounds = {"controller.solve.cost.R": (1e-3, None)}  # R stays a weight: positive
best = Optimizer(
    double_evaluator.program(wrt, bounds=bounds, params=start),
    method="adam",
    n_iterations=20,
).solve()
print(best)

The whole closed loop, network and solver included, is still a system: linearized by
JAX at its nominal state, it has a frequency response in $z$, here from the setpoint $r$ to the
angle.

In [ ]:
from minilink.analysis import plot_bode

double_loop = double_evaluator.loop
plot_bode(double_loop, of=("y", 0), wrt=("r", 0), method="jax", margins=False)

## Summary

| Need | Code |
| --- | --- |
| A rollout as a function of the parameters | `ev = loop.compile(backend="jax")`; `ev.rollout(x0, u_seq, t0, dt, params, n_steps=)` |
| Its gradient | `jax.value_and_grad(rollout_cost)(params)` |
| A family of plants | `ev.rollout_batch(starts, n_steps=, dt=, params=family)` |
| Co-design on a problem | `problem.compile(law % dt, backend="jax").program(wrt, bounds=)`, then `Optimizer` |
| A solver inside the loop | a native planner (`method="adam"`, `n_iterations=`) in `ModelPredictiveController` |